In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import polars as pl
from data import build_dataset
from models import (
    GlobalMeanModel,
    ItemMeanModel,
    UserMeanModel,
    BiasModel,
    PopularityModel,
    MFSVDModel,
    ALSModel,
)
from evaluate import evaluate_rating_model, evaluate_ranking_model

In [2]:
ds = build_dataset(k=10)

print(f"Train:      {ds.n_train:>10,}")
print(f"Val:        {ds.n_val:>10,}")
print(f"Test:       {ds.n_test:>10,}")
print(f"Users:      {ds.n_users:>10,}")
print(f"Items:      {ds.n_items:>10,}")
print(f"Sparsity:   {ds.sparsity * 100:.4f}%")

Train:      25,474,155
Val:           470,882
Test:          308,390
Users:         170,111
Items:          27,929
Sparsity:   99.4638%


In [3]:
K = 10

mf_model = MFSVDModel(n_users=ds.n_users, n_items=ds.n_items, k=50)

rating_models = {
    "GlobalMean":   GlobalMeanModel(),
    "ItemMean":     ItemMeanModel(),
    "UserMean":     UserMeanModel(),
    "Bias (ALS)":   BiasModel(n_epochs=10, lambda_=25.0),
    "MFSVD":        mf_model,
}

ranking_models = {
    "Popularity":   PopularityModel(),
    "MFSVD":        mf_model,
}

In [4]:
# Fit each unique model instance exactly once
fitted = set()
for name, model in {**rating_models, **ranking_models}.items():
    if id(model) not in fitted:
        print(f"Fitting {name} ")
        model.fit(ds.train_df)
        fitted.add(id(model))
        print(" Done.")

Fitting GlobalMean 
 Done.
Fitting ItemMean 
 Done.
Fitting UserMean 
 Done.
Fitting Bias (ALS) 
 Done.
Fitting MFSVD 
 Done.
Fitting Popularity 
 Done.


In [5]:
rating_results = {}

for name, model in rating_models.items():
    rating_results[name] = evaluate_rating_model(model, ds.val_df)
    print(f"{name:<15} RMSE: {rating_results[name]['rmse']:.4f} | "
          f"MAE: {rating_results[name]['mae']:.4f}")

GlobalMean      RMSE: 1.0251 | MAE: 0.7762
ItemMean        RMSE: 0.9610 | MAE: 0.7212
UserMean        RMSE: 0.9744 | MAE: 0.7271
Bias (ALS)      RMSE: 0.8734 | MAE: 0.6444
MFSVD           RMSE: 0.9813 | MAE: 0.7395


In [6]:
ranking_results = {}

for name, model in ranking_models.items():
    ranking_results[name] = evaluate_ranking_model(
        model=model,
        train_df=ds.train_df,
        eval_df=ds.val_df,
        k=K,
        relevance_threshold=4.0,
        item_popularity=ds.item_popularity,
        n_items=ds.n_items,
    )
    print(f"{name:<15} NDCG@{K}: {ranking_results[name][f'ndcg@{K}']:.4f} | "
          f"Recall@{K}: {ranking_results[name][f'recall@{K}']:.4f}")

Popularity      NDCG@10: 0.0746 | Recall@10: 0.0283
MFSVD           NDCG@10: 0.1293 | Recall@10: 0.0508


In [7]:
rating_table = (
    pl.DataFrame([
        {"model": name, **metrics}
        for name, metrics in rating_results.items()
    ])
    .select(["model", "rmse", "mae"])
    .sort("rmse")
)

display(rating_table)

model,rmse,mae
str,f64,f64
"""Bias (ALS)""",0.873443,0.644368
"""ItemMean""",0.960998,0.721199
"""UserMean""",0.974353,0.727107
"""MFSVD""",0.981282,0.739483
"""GlobalMean""",1.025104,0.776174


In [8]:
ranking_table = (
    pl.DataFrame([
        {"model": name, **metrics}
        for name, metrics in ranking_results.items()
    ])
    .select([
        "model",
        f"ndcg@{K}",
        f"recall@{K}",
        f"precision@{K}",
        f"hit_rate@{K}",
        f"map@{K}",
        f"mrr@{K}",
        "coverage",
        "novelty",
        "n_eval_users",
    ])
    .sort(f"ndcg@{K}", descending=True)
)

display(ranking_table)

model,ndcg@10,recall@10,precision@10,hit_rate@10,map@10,mrr@10,coverage,novelty,n_eval_users
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""MFSVD""",0.129306,0.050828,0.111636,0.475589,0.069754,0.252035,0.040209,10.018553,6411.0
"""Popularity""",0.074623,0.028319,0.062143,0.314927,0.037953,0.160133,0.010849,8.822389,6411.0


## Observations

### Rating Prediction (RMSE / MAE)

| Pattern | Finding |
|---|---|
| ItemMean < UserMean | Item variance dominates user variance, movies cluster more strongly by quality than users cluster by taste. Expected on this dataset given the severe long tail. |
| Bias (ALS) best RMSE | Capturing user and item biases simultaneously with L2 regularisation gives a meaningful gain (~0.09 RMSE over ItemMean). Bias terms absorb the bulk of predictable signal at this sparsity level. |
| MFSVD worse than Bias | SVD performs low-rank matrix reconstruction, it is not directly optimising RMSE, nor does it incorporate explicit bias terms or regularisation. BiasModel and MFSVD are solving fundamentally different objective functions. Comparing them on RMSE alone is asymmetric. A regularised MF that incorporates bias terms (SVD++, ALS) may close this gap. |

### Ranking (NDCG@10 / Recall@10)

| Pattern | Finding |
|---|---|
| MFSVD >> Popularity on NDCG | Personalisation meaningfully improves ranking quality. MFSVD's NDCG@10 is 73% higher than Popularity (0.129 vs 0.075). |
| Popularity coverage = 0.011 | Popularity recommends the same ~250 items to every user out of 23K+ in the filtered catalog, a direct materialisation of the long tail problem seen in EDA. |
| MFSVD coverage = 0.040 | 3.7x broader catalog coverage than Popularity. Latent factors surface niche items that raw interaction counts never reach. |
| MFSVD novelty > Popularity | Confirms MFSVD is not purely recommending blockbusters. Self-information score of 10.02 vs 8.82. |
| Low absolute NDCG values | Expected given evaluation strictness: K=10, relevance threshold=4.0, temporal split producing thin val sets per user. The upper bound on recall is inherently low. The system is not failing, the evaluation setup is strict. |

### Evaluation Notes

Ranking metrics are computed only on users with at least one relevant item
(rating ≥ 4.0) in the validation split, 6,411 users (~3.7% of total). This
is standard practice in recommender evaluation but should be noted explicitly.
The low coverage is a temporal split thinness problem, most users do not have
enough interactions in any fixed time window to produce dense per-user ground
truth. Lowering the relevance threshold to 3.5 produced negligible improvement
(6,525 users), confirming the threshold is not the bottleneck. Per-user
leave-last-N splitting will replace the global temporal cutoff in later
evaluation stages to produce denser per-user ground truth.

### Key Findings

1. **BiasModel achieves the best RMSE**, confirming that regularised bias terms
   dominate rating prediction at high sparsity.
2. **MFSVD underperforms on RMSE but significantly improves ranking metrics**,
   highlighting the central tradeoff: optimising for rating prediction accuracy
   does not optimise for ranking quality. These are different objectives.
3. **BiasModel personalises via per-user bias terms** but lacks latent
   collaborative structure. MFSVD captures latent structure but lacks explicit
   bias terms and regularisation. Neither is complete, SVD++ or regularised
   ALS addresses both simultaneously.
4. **Popularity is a deceptively strong recall baseline** but collapses
   recommendations into a narrow head of the catalog, as shown by coverage
   (0.011) and novelty (8.82).
5. **MFSVD improves both coverage and novelty**, indicating better exploration
   of the item space at the cost of model complexity.

In [3]:
ds_lln = build_dataset(split="leave_last_n", k=10, lln_n=2)

print(f"LLN Train:          {ds_lln.n_train:>10,}")
print(f"LLN Val:            {ds_lln.n_val:>10,}")
print(f"LLN Test:           {ds_lln.n_test:>10,}")
print(f"LLN Users:          {ds_lln.n_users:>10,}")
print(f"LLN Items:          {ds_lln.n_items:>10,}")
print(f"LLN Eval users:     {ds_lln.n_eval_users:>10,}")
print(f"LLN Eval coverage:  {ds_lln.eval_coverage:.2%}")
print(f"LLN Sparsity:       {ds_lln.sparsity * 100:.4f}%")

LLN Train:          31,038,908
LLN Val:               401,894
LLN Test:              401,894
LLN Users:             200,947
LLN Items:              31,961
LLN Eval users:        200,947
LLN Eval coverage:  100.00%
LLN Sparsity:       99.5167%


In [10]:
lln_ranking_results = {}

pop_lln = PopularityModel()
mf_lln  = MFSVDModel(n_users=ds_lln.n_users, n_items=ds_lln.n_items, k=50)

lln_ranking_models = {
    "Popularity":   pop_lln,
    "MFSVD":        mf_lln,
}

for name, model in lln_ranking_models.items():
    print(f"Fitting {name}...")
    model.fit(ds_lln.train_df)
    print(" Done.")

for name, model in lln_ranking_models.items():
    lln_ranking_results[name] = evaluate_ranking_model(
        model=model,
        train_df=ds_lln.train_df,
        eval_df=ds_lln.val_df,
        k=K,
        relevance_threshold=4.0,
        item_popularity=ds_lln.item_popularity,
        n_items=ds_lln.n_items,
    )
    print(f"{name:<15} NDCG@{K}: {lln_ranking_results[name][f'ndcg@{K}']:.4f} | "
          f"Recall@{K}: {lln_ranking_results[name][f'recall@{K}']:.4f} | "
          f"n_eval_users: {lln_ranking_results[name]['n_eval_users']:.0f}")

Fitting Popularity...
 Done.
Fitting MFSVD...
 Done.
Popularity      NDCG@10: 0.0323 | Recall@10: 0.0570 | n_eval_users: 152244
MFSVD           NDCG@10: 0.0648 | Recall@10: 0.1073 | n_eval_users: 152244


In [11]:
lln_validation_table = (
    pl.DataFrame([
        {"model": name, **metrics}
        for name, metrics in lln_ranking_results.items()
    ])
    .select([
        "model",
        f"ndcg@{K}",
        f"recall@{K}",
        f"precision@{K}",
        f"hit_rate@{K}",
        f"map@{K}",
        f"mrr@{K}",
        "coverage",
        "novelty",
        "n_eval_users",
    ])
    .sort(f"ndcg@{K}", descending=True)
)

display(lln_validation_table)

model,ndcg@10,recall@10,precision@10,hit_rate@10,map@10,mrr@10,coverage,novelty,n_eval_users
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""MFSVD""",0.064791,0.107321,0.017176,0.156321,0.043538,0.063588,0.043929,9.748596,152244.0
"""Popularity""",0.032264,0.056974,0.009049,0.083281,0.020619,0.030314,0.011107,8.715752,152244.0


In [ ]:
# K=10

In [12]:
als_model = ALSModel(factors=50, iterations=20, regularization=0.01)
als_model.fit(ds_lln.train_df, ds_lln.implicit_matrix)

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
als_model_fixed = ALSModel(factors=128, iterations=50, regularization=0.01, alpha=40.0)
als_model_fixed.fit(ds_lln.train_df, ds_lln.implicit_matrix)


  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
als_fixed_results = evaluate_ranking_model(
    model=als_model_fixed,
    train_df=ds_lln.train_df,
    eval_df=ds_lln.val_df,
    k=K,
    relevance_threshold=4.0,
    item_popularity=ds_lln.item_popularity,
    n_items=ds_lln.n_items,
)

NameError: name 'K' is not defined

In [ ]:
print(f"ALS NDCG@{K}:   {als_fixed_results[f'ndcg@{K}']:.6f} | "
      f"Recall@{K}:     {als_fixed_results[f'recall@{K}']:.6f} | "
      f"n_eval_users:   {als_fixed_results['n_eval_users']:.0f}")

ALS NDCG@10:   0.067645 | Recall@10:     0.122737 | n_eval_users:   152244


In [13]:
als_results = evaluate_ranking_model(
    model=als_model,
    train_df=ds_lln.train_df,
    eval_df=ds_lln.val_df,
    k=K,
    relevance_threshold=4.0,
    item_popularity=ds_lln.item_popularity,
    n_items=ds_lln.n_items,
)

print(f"ALS NDCG@{K}:   {als_results[f'ndcg@{K}']:.6f} | "
      f"Recall@{K}:     {als_results[f'recall@{K}']:.6f} | "
      f"n_eval_users:   {als_results['n_eval_users']:.0f}")

ALS NDCG@10:   0.032744 | Recall@10:     0.062837 | n_eval_users:   152244


In [14]:
lln_ranking_results["ALS"] = als_results

lln_final_table = (
    pl.DataFrame([
        {"model": name, **metrics}
        for name, metrics in lln_ranking_results.items()
    ])
    .select([
        "model",
        f"ndcg@{K}",
        f"recall@{K}",
        f"precision@{K}",
        f"hit_rate@{K}",
        f"map@{K}",
        f"mrr@{K}",
        "coverage",
        "novelty",
        "n_eval_users",
    ])
    .sort(f"ndcg@{K}", descending=True)
)

display(lln_final_table)

model,ndcg@10,recall@10,precision@10,hit_rate@10,map@10,mrr@10,coverage,novelty,n_eval_users
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""MFSVD""",0.064791,0.107321,0.017176,0.156321,0.043538,0.063588,0.043929,9.748596,152244.0
"""ALS""",0.032744,0.062837,0.009644,0.090046,0.019826,0.02824,0.278026,11.283655,152244.0
"""Popularity""",0.032264,0.056974,0.009049,0.083281,0.020619,0.030314,0.011107,8.715752,152244.0


In [15]:
print(f"implicit_matrix nnz: {ds_lln.implicit_matrix.nnz:,}")
print(f"implicit_matrix max value: {ds_lln.implicit_matrix.max():.2f}")
print(f"implicit_matrix min value: {ds_lln.implicit_matrix.min():.2f}")

implicit_matrix nnz: 31,038,908
implicit_matrix max value: 201.00
implicit_matrix min value: 0.00


In [16]:
print(f"user_factors shape: {als_model._user_factors.shape}")
print(f"item_factors shape: {als_model._item_factors.shape}")
print(f"expected: Users: ({ds_lln.n_users}, 50) and Items: ({ds_lln.n_items}, 50)")

user_factors shape: (200947, 50)
item_factors shape: (31961, 50)
expected: Users: (200947, 50) and Items: (31961, 50)


### ALS (Implicit Feedback), LLN Evaluation

- ALS marginally beats Popularity on NDCG and Recall despite being trained on implicit confidence signals rather than explicit ratings, confirms the model is learning meaningful preference structure.
- ALS coverage (27.8%) dwarfs both MFSVD (4.4%) and Popularity (1.1%), reflecting the implicit model's ability to surface long-tail items that explicit rating models and popularity counts miss entirely.
- ALS novelty (11.28) is the highest of all models, consistent with its broader catalog exploration.
- ALS underperforms MFSVD on accuracy metrics when evaluated against explicit relevance (rating ≥ 4.0). This is expected, ALS optimises a different objective (confidence-weighted ranking) and is penalised by an evaluation protocol designed for explicit feedback models.
- Direct ALS vs MFSVD comparison is somewhat unfair given the objective mismatch. A ranking-native evaluation (BPR loss, sampled softmax) would give a cleaner comparison.
  
- Note: The underperformance relative to MFSVD is most likely primarily an objective mismatch, ALS is optimised for confidence-weighted implicit ranking while evaluation uses explicit relevance (rating ≥ 4.0). Hyperparameter tuning (factors, iterations, regularization) is deferred to a dedicated tuning pass.